# Chapter 6 — Information Retrieval and Knowledge Agents

**Book:** AI Agents (Packt, 2026)
**Author:** Imran Ahmad

---

This notebook implements the three knowledge agent types covered in Chapter 6:

1. **Knowledge Retrieval Agent (§6.1)** — RAG pipeline with LangChain, FAISS, and OpenAI
2. **Document Intelligence Agent (§6.2)** — OCR-based schema extraction from invoices
3. **Scientific Research Agent (§6.3)** — Literature scanning, clustering, and synthesis

**Execution Modes:** The notebook detects whether an OpenAI API key is available.
If no key is found, it activates **Simulation Mode**, replacing all external calls
with chapter-derived mock outputs. Both modes produce pedagogically equivalent results.

## 0. Setup & Configuration

This section loads dependencies, initializes the resilience layer, and detects the
execution mode.

| Mode | Trigger | Behavior |
|------|---------|----------|
| **Live Mode** | Valid `OPENAI_API_KEY` in `.env` | Full API calls to OpenAI, live arXiv queries |
| **Simulation Mode** | No API key (default) | Chapter-derived mocks from `agent_utils.py` |

In [ ]:
# ─────────────────────────────────────────────
# 0.1 Dependency Check
# Ref: requirements.txt
# ─────────────────────────────────────────────

import importlib, sys

REQUIRED = [
    "dotenv", "numpy", "pandas",       # Core
    "langchain", "faiss",              # §6.1 RAG
    "PIL", "rapidfuzz",                # §6.2 Document Intelligence
]

missing = []
for pkg in REQUIRED:
    try:
        importlib.import_module(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"⚠️  Missing packages: {missing}")
    print("   Run: pip install -r requirements.txt")
else:
    print("✅ All core dependencies available.")

In [ ]:
# ─────────────────────────────────────────────
# 0.2 Imports & Resilience Layer
# Ref: agent_utils.py (ColorLogger, fail_gracefully, mocks)
# ─────────────────────────────────────────────

import os
import sys
import numpy as np
import pandas as pd

# Ensure the notebook can find agent_utils.py
sys.path.insert(0, os.path.abspath("."))

from agent_utils import (
    ColorLogger, log, fail_gracefully, get_api_key,
    MockLLM, MockRetrievalQAResult, MockEmbeddings,
    MockOcrToken, MOCK_INVOICE_TOKENS, MOCK_EXTRACTED_FIELDS,
    mock_pytesseract_output, MOCK_ARXIV_PAPERS, mock_search_arxiv,
)

log.info("agent_utils loaded — resilience layer active.")

In [ ]:
# ─────────────────────────────────────────────
# 0.3 API Key Detection & Mode Selection
# Ref: agent_utils.get_api_key() — cascading fallback
# ─────────────────────────────────────────────

api_key = get_api_key("OPENAI_API_KEY")
SIMULATION_MODE = api_key is None

# ── Mode Banner ──
if SIMULATION_MODE:
    print("\n" + "=" * 64)
    print("  🧪 SIMULATION MODE — No API key detected.")
    print("  All outputs are chapter-derived mocks from agent_utils.py.")
    print("  To enable Live Mode, add OPENAI_API_KEY to .env")
    print("=" * 64 + "\n")
else:
    print("\n" + "=" * 64)
    print("  🟢 LIVE MODE — API key detected.")
    print("  External API calls will be made to OpenAI and other services.")
    print("=" * 64 + "\n")

---

## 1. Knowledge Retrieval Agent (§6.1)

A Knowledge Retrieval agent connects an LLM's static training data to live, verifiable
information sources. It addresses two critical weaknesses of language models: **knowledge
cutoff** and the **risk of hallucination**, anchoring outputs in evidence.

### Architecture (Ref: Figure 6.1, p.4)

The retrieval process follows four modular stages:

1. **Query Understanding** — Parse intent, disambiguate, reformulate into a search query
2. **Retrieval** — Plan strategy, execute via search APIs or vector databases
3. **Preprocessing** — Chunk documents, generate embeddings, filter irrelevant results
4. **Synthesis** — Integrate retrieved content into the LLM prompt; generate a grounded answer

A parallel **Provenance** component collects citations and confidence metrics throughout
the pipeline, ensuring the final answer is auditable.

In this section, we implement a complete RAG pipeline using LangChain, OpenAI embeddings
(or mock equivalents), and FAISS as the vector store.

### 1.1 Document Loading

We load the synthetic corpus from the `docs/` directory. This corpus contains:
- `knowledge_base_rag.txt` — RAG concepts, strategies, chunking, and operational challenges (~900 words)
- `compliance_policy.txt` — Corporate data retention and refund policy (~400 words)

These documents serve as the knowledge base that the retrieval agent will search.

In [ ]:
# ─────────────────────────────────────────────
# 1.1 Load Documents
# Ref: Listing on p.6 — DirectoryLoader("./docs", glob="*.txt")
# ─────────────────────────────────────────────

from langchain_community.document_loaders import DirectoryLoader, TextLoader

@fail_gracefully(fallback_return=lambda: [], section_ref="6.1")
def load_documents(doc_dir="./docs"):
    loader = DirectoryLoader(doc_dir, glob="*.txt", loader_cls=TextLoader)
    docs = loader.load()
    log.info(f"Loaded {len(docs)} documents from '{doc_dir}'.")
    for doc in docs:
        log.info(f"  → {doc.metadata.get('source', 'unknown')} ({len(doc.page_content)} chars)")
    return docs

documents = load_documents()
if not documents:
    log.error("No documents loaded. Check the docs/ directory.")

### 1.2 Text Splitting

We use `RecursiveCharacterTextSplitter` with `chunk_size=1000` and `chunk_overlap=200`,
the recommended default from §6.1 (p.8). Recursive chunking splits on natural boundaries
(paragraphs, sentences, words) in descending order, producing semantically coherent chunks.

The 200-character overlap ensures that a sentence spanning two chunks is fully captured
by at least one of them.

In [ ]:
# ─────────────────────────────────────────────
# 1.2 Split Documents into Chunks
# Ref: p.6 — RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# ─────────────────────────────────────────────

from langchain_text_splitters import RecursiveCharacterTextSplitter

@fail_gracefully(fallback_return=lambda: [], section_ref="6.1")
def split_documents(docs, chunk_size=1000, chunk_overlap=200):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunks = splitter.split_documents(docs)
    log.info(f"Split {len(docs)} documents into {len(chunks)} chunks "
             f"(size={chunk_size}, overlap={chunk_overlap}).")
    return chunks

chunks = split_documents(documents)
if chunks:
    log.success(f"Chunking complete: {len(chunks)} chunks ready for embedding.")
    print(f"\nSample chunk (first 200 chars):\n{chunks[0].page_content[:200]}...")

### 1.3 Embedding & Vector Store

We create vector embeddings and store them in a **FAISS** index (Facebook AI Similarity
Search). In Live Mode, this uses OpenAI's `text-embedding-3-large` model. In Simulation
Mode, `MockEmbeddings` generates deterministic pseudo-vectors using a seeded hash of each
text (Ref: `agent_utils.py`, §5).

FAISS is a highly efficient library for similarity search, making it ideal for production
RAG systems. The `from_documents` method handles both embedding generation and index
creation in a single operation (Ref: p.7).

In [ ]:
# ─────────────────────────────────────────────
# 1.3 Create Embeddings & FAISS Index
# Ref: p.7 — FAISS.from_documents(chunks, embeddings)
# ─────────────────────────────────────────────

from langchain_community.vectorstores import FAISS

@fail_gracefully(fallback_return=None, section_ref="6.1")
def build_vectorstore(chunks, simulation_mode):
    if simulation_mode:
        log.info("[SIMULATION MODE] Using MockEmbeddings (dimension=256).")
        embeddings = MockEmbeddings()
    else:
        from langchain_openai import OpenAIEmbeddings
        embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
        log.info("Using OpenAI text-embedding-3-large.")

    vectorstore = FAISS.from_documents(chunks, embeddings)
    log.info(f"FAISS index built with {len(chunks)} vectors.")
    return vectorstore

vectorstore = build_vectorstore(chunks, SIMULATION_MODE)

### 1.4 Retrieval & Generation Chain

We configure a `RetrievalQA` chain that:
1. Converts the vector store into a **retriever** returning the top 3 most similar chunks (`k=3`)
2. Passes those chunks to a `ChatOpenAI` instance (or `MockLLM` in Simulation Mode)
3. Returns both the generated answer and the source documents for provenance

In Simulation Mode, `MockLLM` routes queries to pre-written responses that teach the same
concepts as live API output (Ref: `agent_utils.py`, §4).

In [ ]:
# ─────────────────────────────────────────────
# 1.4 Build Retrieval + Generation Chain
# Ref: pp.6-7 — RetrievalQA.from_chain_type(llm, retriever, return_source_documents)
# ─────────────────────────────────────────────

@fail_gracefully(fallback_return=None, section_ref="6.1")
def build_qa_chain(vectorstore, simulation_mode):
    if simulation_mode or vectorstore is None:
        log.info("[SIMULATION MODE] Using MockLLM for question answering.")
        return "mock"
    else:
        from langchain_openai import ChatOpenAI
        from langchain.chains import RetrievalQA

        retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            retriever=retriever,
            return_source_documents=True,
        )
        log.info("RetrievalQA chain built with gpt-4o-mini and top-3 retrieval.")
        return qa_chain

qa_chain = build_qa_chain(vectorstore, SIMULATION_MODE)

### 1.5 Run a Query

We submit the query from the chapter: *"What are the main limitations of retrieval-augmented
generation?"* (Ref: p.7). The system embeds the question, retrieves the three most
semantically relevant chunks, and generates an answer grounded in the retrieved information.

The response includes both the answer and source attribution, showing exactly which
documents were used.

In [ ]:
# ─────────────────────────────────────────────
# 1.5 Query the RAG Pipeline
# Ref: p.7 — query = "What are the main limitations of retrieval-augmented generation?"
# ─────────────────────────────────────────────

QUERY = "What are the main limitations of retrieval-augmented generation?"

@fail_gracefully(fallback_return=lambda: MockRetrievalQAResult(QUERY).run(), section_ref="6.1")
def run_query(qa_chain, query, simulation_mode):
    if simulation_mode or qa_chain == "mock":
        log.info("[SIMULATION MODE] Generating mock RAG response.")
        return MockRetrievalQAResult(query).run()
    else:
        return qa_chain({"query": query})

result = run_query(qa_chain, QUERY, SIMULATION_MODE)

# ── Display Results ──
print("\n" + "─" * 64)
print(f"Query: {result['query']}")
print("─" * 64)
print(f"\nAnswer:\n{result['result']}")
print("\nSources:")
for doc in result["source_documents"]:
    source = doc["metadata"]["source"] if isinstance(doc, dict) else doc.metadata.get("source", "unknown")
    print(f"  → {source}")
print("─" * 64)

### 1.6 Diagnostic Scenario: Subscription Refund Query

The chapter (§6.1, p.9) describes a diagnostic scenario: a user asks *"What is our
refund policy for subscriptions?"* and the pipeline returns generic billing terms instead
of the specific subscription clause.

Let's run this query and inspect the source documents to practice the retrieval failure
diagnosis pattern.

In [ ]:
# ─────────────────────────────────────────────
# 1.6 Diagnostic Query — Refund Policy
# Ref: §6.1 p.9 — "What is our refund policy for subscriptions?"
# ─────────────────────────────────────────────

DIAG_QUERY = "What is our refund policy for subscriptions?"

diag_result = run_query(qa_chain, DIAG_QUERY, SIMULATION_MODE)

print("\n" + "─" * 64)
print(f"Diagnostic Query: {diag_result['query']}")
print("─" * 64)
print(f"\nAnswer:\n{diag_result['result']}")
print("\nSources (inspect for correct routing):")
for doc in diag_result["source_documents"]:
    source = doc["metadata"]["source"] if isinstance(doc, dict) else doc.metadata.get("source", "unknown")
    print(f"  → {source}")
print("─" * 64)

log.info(
    "Diagnostic pattern: If the sources above point to the generic FAQ rather than "
    "the subscription policy, the fix is to re-ingest the missing document, adjust "
    "chunk_size, or add a metadata filter (§6.1, p.10)."
)

---

## 2. Chunking Strategies Deep Dive (§6.1)

Chunking is the **most consequential configuration decision** in a RAG system (§6.1, p.8).
A chunk is the atomic unit retrieved by the vector index; its size determines how much
text the LLM receives as context for each match.

This section demonstrates the three chunking strategies described in the chapter:

| Strategy | Description | Best For |
|----------|-------------|----------|
| **Fixed-size** | Splits at a fixed character boundary | Uniform, well-formatted documents |
| **Recursive** | Splits on natural boundaries (paragraphs → sentences → words) | Mixed-content corpora (recommended default) |
| **Semantic** | Uses embedding similarity to detect topic shifts | Narrative text (higher computational cost) |

### The Size-Overlap Trade-Off

- **Smaller chunks (200–500 chars):** Better retrieval precision, but risk omitting context
- **Larger chunks (1,000–2,000 chars):** Richer context, but diluted embedding signal
- **Overlap (e.g., 200 chars):** Ensures sentences spanning two chunks are captured by at least one

In [ ]:
# ─────────────────────────────────────────────
# 2.1 Sample Text for Chunking Comparison
# ─────────────────────────────────────────────

# We use the RAG knowledge base as a realistic sample corpus
with open("docs/knowledge_base_rag.txt", "r") as f:
    SAMPLE_TEXT = f.read()

log.info(f"Sample text loaded: {len(SAMPLE_TEXT)} characters, ~{len(SAMPLE_TEXT.split())} words.")

In [ ]:
# ─────────────────────────────────────────────
# 2.2 Fixed-Size Chunking
# Ref: §6.1 p.8 — "divides text at a fixed character or token boundary"
# ─────────────────────────────────────────────

from langchain_text_splitters import CharacterTextSplitter

fixed_splitter = CharacterTextSplitter(
    separator="",           # Pure character-level split
    chunk_size=500,
    chunk_overlap=100,
    is_separator_regex=False,
)
fixed_chunks = fixed_splitter.split_text(SAMPLE_TEXT)

log.info(f"Fixed-size chunking: {len(fixed_chunks)} chunks (size=500, overlap=100)")
for i, chunk in enumerate(fixed_chunks[:3]):
    print(f"  Chunk {i}: {len(chunk)} chars — '{chunk[:60]}...'")

In [ ]:
# ─────────────────────────────────────────────
# 2.3 Recursive Chunking (Recommended Default)
# Ref: §6.1 p.8 — "attempts to split on natural boundaries
#   (paragraphs, sentences, words) in descending order"
# Ref: p.6 — chunk_size=1000, chunk_overlap=200
# ─────────────────────────────────────────────

from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,         # Same size as fixed for fair comparison
    chunk_overlap=100,
)
recursive_chunks = recursive_splitter.split_text(SAMPLE_TEXT)

log.info(f"Recursive chunking: {len(recursive_chunks)} chunks (size=500, overlap=100)")
for i, chunk in enumerate(recursive_chunks[:3]):
    print(f"  Chunk {i}: {len(chunk)} chars — '{chunk[:60]}...'")

In [ ]:
# ─────────────────────────────────────────────
# 2.4 Semantic Chunking (Concept Demonstration)
# Ref: §6.1 p.8 — "uses embedding similarity to detect natural
#   topic shifts before splitting"
# ─────────────────────────────────────────────

# Semantic chunking requires embeddings to detect topic boundaries.
# Here we demonstrate the concept by splitting on section headers
# (a lightweight proxy for topic-shift detection).

import re

def semantic_chunking_demo(text):
    """
    Simplified semantic chunking: split on numbered section headers
    (e.g., '1. What Is RAG', '2. The Retrieval Process').
    In production, this would use embedding similarity to detect shifts.
    """
    # Split on lines that start with a digit followed by a period
    sections = re.split(r'\n(?=\d+\.\s)', text)
    # Filter out empty sections
    sections = [s.strip() for s in sections if s.strip()]
    return sections

semantic_chunks = semantic_chunking_demo(SAMPLE_TEXT)

log.info(f"Semantic chunking (topic-based): {len(semantic_chunks)} chunks")
for i, chunk in enumerate(semantic_chunks):
    # Extract the first line as a section title
    title = chunk.split("\n")[0][:70]
    print(f"  Chunk {i}: {len(chunk)} chars — '{title}...'")

In [ ]:
# ─────────────────────────────────────────────
# 2.5 Comparison Summary
# Ref: §6.1 p.8 — size-overlap trade-off analysis
# ─────────────────────────────────────────────

comparison = pd.DataFrame({
    "Strategy": ["Fixed-size", "Recursive", "Semantic (topic-based)"],
    "Chunks": [len(fixed_chunks), len(recursive_chunks), len(semantic_chunks)],
    "Avg Chars": [
        int(np.mean([len(c) for c in fixed_chunks])),
        int(np.mean([len(c) for c in recursive_chunks])),
        int(np.mean([len(c) for c in semantic_chunks])),
    ],
    "Min Chars": [
        min(len(c) for c in fixed_chunks),
        min(len(c) for c in recursive_chunks),
        min(len(c) for c in semantic_chunks),
    ],
    "Max Chars": [
        max(len(c) for c in fixed_chunks),
        max(len(c) for c in recursive_chunks),
        max(len(c) for c in semantic_chunks),
    ],
})

print("\n" + "=" * 64)
print("  CHUNKING STRATEGY COMPARISON")
print("=" * 64)
print(comparison.to_string(index=False))
print("=" * 64)
print()
log.info(
    "Key insight (§6.1, p.8): Recursive chunking is the recommended default "
    "for mixed-content corpora. It produces more semantically coherent chunks "
    "than fixed-size splitting while avoiding the computational overhead of "
    "full semantic chunking."
)
log.info(
    "The common default of chunk_size=1000 with chunk_overlap=200 ensures "
    "that sentences spanning two chunks are fully captured by at least one."
)

---

## 3. Document Intelligence Agent (§6.2)

A Document Intelligence agent moves beyond simple retrieval to **understand, parse, and
transform** messy, unstructured documents into clean, trustworthy, and machine-readable
data (§6.2, p.10).

### The Five-Stage Pipeline (Ref: Figure 6.2, p.17)

| Stage | Function | Key Metric |
|-------|----------|------------|
| 1. **Ingest & Triage** | Classify document type, detect language, route to workflow | MIME-type + classifier |
| 2. **OCR & Preprocess** | Deskew, denoise, convert images to text with confidence scores | Confidence scoring |
| 3. **Layout Parse** | Identify headings, paragraphs, tables, key-value pairs | Reading order |
| 4. **Extract Data** | Schema-driven entity and relationship extraction | 95%+ accuracy target |
| 5. **Validate & Integrate** | Route by confidence: auto-integrate or flag for HITL review | <8% human review |

In this section, we:
1. Generate a synthetic invoice image using Pillow
2. Run OCR (real Tesseract or mock) with confidence scoring
3. Apply the `CONFIDENCE_THRESHOLD=60` filter from §6.2 (p.12)
4. Extract fields using the `SCHEMA` pattern from §6.2 (p.13)

### 3.1 Generate Synthetic Invoice

We use Pillow to programmatically create a realistic invoice PNG. This ensures the
demo is fully self-contained — no external image downloads required.

The invoice contains:
- Header: "INVOICE"
- Invoice No: INV-2026-00142
- Date: 2026-03-15
- Line items with descriptions and amounts
- Total Due: $4,750.00
- One deliberately smudged region to demonstrate confidence thresholding

In [ ]:
# ─────────────────────────────────────────────
# 3.1 Generate Synthetic Invoice Image
# Ref: §6.2 — Strategy Part II-F specification
# ─────────────────────────────────────────────

import os
from PIL import Image, ImageDraw, ImageFont, ImageFilter

@fail_gracefully(fallback_return="samples/sample_invoice.png", section_ref="6.2")
def generate_invoice(output_path="samples/sample_invoice.png"):
    """Create a synthetic invoice PNG using Pillow."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    W, H = 600, 500
    img = Image.new("RGB", (W, H), "white")
    draw = ImageDraw.Draw(img)

    # Use default font (available everywhere)
    try:
        font_large = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 24)
        font_med = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 14)
        font_small = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 12)
    except (IOError, OSError):
        font_large = ImageFont.load_default()
        font_med = ImageFont.load_default()
        font_small = ImageFont.load_default()

    # Header
    draw.text((50, 30), "INVOICE", fill="black", font=font_large)
    draw.line([(50, 60), (550, 60)], fill="black", width=2)

    # Invoice details
    draw.text((50, 80), "Invoice No:", fill="gray", font=font_med)
    draw.text((170, 80), "INV-2026-00142", fill="black", font=font_med)
    draw.text((50, 105), "Date:", fill="gray", font=font_med)
    draw.text((170, 105), "2026-03-15", fill="black", font=font_med)
    draw.text((50, 130), "Bill To:", fill="gray", font=font_med)
    draw.text((170, 130), "Acme Corporation", fill="black", font=font_med)

    # Line items table
    y_table = 175
    draw.line([(50, y_table), (550, y_table)], fill="black", width=1)
    draw.text((50, y_table + 5), "Description", fill="black", font=font_med)
    draw.text((350, y_table + 5), "Qty", fill="black", font=font_med)
    draw.text((420, y_table + 5), "Unit Price", fill="black", font=font_med)
    draw.text((500, y_table + 5), "Amount", fill="black", font=font_med)
    draw.line([(50, y_table + 25), (550, y_table + 25)], fill="black", width=1)

    items = [
        ("AI Agent Development Services", "50", "$75.00", "$3,750.00"),
        ("Cloud Infrastructure Setup", "1", "$600.00", "$600.00"),
        ("Documentation & Training", "8", "$50.00", "$400.00"),
    ]
    for i, (desc, qty, unit, amt) in enumerate(items):
        y = y_table + 35 + i * 25
        draw.text((50, y), desc, fill="black", font=font_small)
        draw.text((355, y), qty, fill="black", font=font_small)
        draw.text((420, y), unit, fill="black", font=font_small)
        draw.text((500, y), amt, fill="black", font=font_small)

    # Total
    y_total = y_table + 120
    draw.line([(400, y_total), (550, y_total)], fill="black", width=2)
    draw.text((350, y_total + 8), "Total Due:", fill="black", font=font_med)
    draw.text((480, y_total + 8), "$4,750.00", fill="black", font=font_med)

    # Smudged region (demonstrates low-confidence OCR tokens)
    y_smudge = y_total + 55
    draw.text((50, y_smudge), "Note: Payment terms NET 30", fill="gray", font=font_small)
    # Create smudge effect: draw text, then blur a region
    smudge_region = img.crop((45, y_smudge - 2, 260, y_smudge + 18))
    smudge_region = smudge_region.filter(ImageFilter.GaussianBlur(radius=3))
    img.paste(smudge_region, (45, y_smudge - 2))

    # Footer
    draw.text((50, H - 40), "Thank you for your business.", fill="gray", font=font_small)

    img.save(output_path)
    log.success(f"Invoice saved to {output_path} ({W}x{H} pixels)")
    return output_path

invoice_path = generate_invoice()

# Display the invoice inline
from IPython.display import display, Image as IPImage
display(IPImage(filename=invoice_path, width=500))

### 3.2 OCR with Confidence Scoring

The OCR stage (Stage 2 of the pipeline) converts printed text into digital characters,
emitting **confidence scores** for each recognized token (Ref: §6.2, p.12).

The chapter specifies `CONFIDENCE_THRESHOLD = 60` — tokens below this threshold are
flagged for human review. This demonstrates the **resilient design** principle: cascading
strategies for low-confidence fields (Ref: §6.2, p.18).

In Simulation Mode, we use `mock_pytesseract_output()` which returns tokens matching
the invoice layout, including one deliberately low-confidence token ("Smudged" at conf=35).

In [ ]:
# ─────────────────────────────────────────────
# 3.2 OCR — Extract Tokens with Confidence Scores
# Ref: §6.2 p.12 — CONFIDENCE_THRESHOLD = 60
# Ref: §6.2 p.12 — pytesseract.image_to_data() output format
# ─────────────────────────────────────────────

CONFIDENCE_THRESHOLD = 60  # Tesseract confidence: 0-100 (Ref: §6.2, p.12)

@fail_gracefully(fallback_return=mock_pytesseract_output, section_ref="6.2")
def run_ocr(image_path, simulation_mode):
    if simulation_mode:
        log.info("[SIMULATION MODE] Using mock OCR tokens.")
        return mock_pytesseract_output()
    else:
        import pytesseract
        from PIL import Image as PILImage
        img = PILImage.open(image_path)
        data = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT)
        log.info(f"Tesseract returned {len(data['text'])} tokens.")
        return data

ocr_data = run_ocr(invoice_path, SIMULATION_MODE)

# ── Display All Tokens with Confidence ──
print("\n" + "=" * 64)
print("  OCR TOKENS (with confidence scores)")
print("=" * 64)
print(f"  {'Token':<20} {'Conf':>6}  {'Status'}")
print("  " + "─" * 50)

high_conf_tokens = []
low_conf_tokens = []

for i, text in enumerate(ocr_data["text"]):
    text = text.strip()
    if not text:
        continue
    conf = int(ocr_data["conf"][i])
    if conf >= CONFIDENCE_THRESHOLD:
        status = "✅ PASS"
        high_conf_tokens.append(text)
    else:
        status = "⚠️  BELOW THRESHOLD"
        low_conf_tokens.append(text)
    print(f"  {text:<20} {conf:>5}%  {status}")

print("  " + "─" * 50)
print(f"  High confidence: {len(high_conf_tokens)} tokens")
print(f"  Low confidence:  {len(low_conf_tokens)} tokens (flagged for HITL review)")
print("=" * 64)

log.info(
    f"Confidence filter applied: {len(high_conf_tokens)} tokens passed "
    f"(threshold={CONFIDENCE_THRESHOLD}), {len(low_conf_tokens)} flagged."
)

### 3.3 Schema-Driven Field Extraction

Stage 4 of the pipeline performs **schema-driven extraction** to pull specific entities
(Ref: §6.2, p.13). The output is structured data in JSON format, complete with
confidence scores and provenance.

The `SCHEMA` from the chapter defines cue keywords for each target field:

```python
SCHEMA = {
    "invoice_number": ["invoice no", "invoice number", "inv no"],
    "invoice_date":   ["date", "invoice date"],
    "total_amount":   ["total", "amount due", "balance due", "total due"],
}
```

The extraction strategy:
1. Find a line containing or closely matching a cue keyword
2. Return the text to the right of that keyword on the same line
3. Fall back to the next line below in the same column region

In [ ]:
# ─────────────────────────────────────────────
# 3.3 Schema-Driven Field Extraction
# Ref: §6.2 pp.13-16 — SCHEMA, extract_near_keyword, fuzzy matching
# ─────────────────────────────────────────────

# Schema definition (Ref: §6.2, p.13)
SCHEMA = {
    "invoice_number": ["invoice no", "invoice number", "inv no"],
    "invoice_date":   ["date", "invoice date"],
    "total_amount":   ["total", "amount due", "balance due", "total due"],
}

@fail_gracefully(fallback_return=lambda: MOCK_EXTRACTED_FIELDS, section_ref="6.2")
def extract_fields(ocr_data, schema, simulation_mode):
    """
    Extract fields from OCR tokens using the schema-driven approach
    from §6.2 (pp.13-16): group tokens by line, find cue keywords,
    return adjacent value text.
    """
    if simulation_mode:
        log.info("[SIMULATION MODE] Returning pre-computed extracted fields.")
        return MOCK_EXTRACTED_FIELDS

    # Group tokens by line
    lines = {}
    for i, text in enumerate(ocr_data["text"]):
        text = text.strip()
        if not text:
            continue
        conf = int(ocr_data["conf"][i])
        if conf < CONFIDENCE_THRESHOLD:
            continue
        line_id = ocr_data["line_num"][i]
        if line_id not in lines:
            lines[line_id] = []
        lines[line_id].append({
            "text": text,
            "x": ocr_data["left"][i],
            "conf": conf,
        })

    # Sort tokens within each line by x-position
    for lid in lines:
        lines[lid] = sorted(lines[lid], key=lambda t: t["x"])

    # Extract fields using cue keyword matching
    from rapidfuzz import fuzz
    results = {}
    for field_name, cues in schema.items():
        results[field_name] = ""
        for lid, tokens in sorted(lines.items()):
            line_text = " ".join(t["text"] for t in tokens).lower()
            for cue in cues:
                if fuzz.partial_ratio(line_text, cue.lower()) >= 80:
                    # Value is the text after the cue keyword
                    full_text = " ".join(t["text"] for t in tokens)
                    # Find the value portion (after colon or last cue word)
                    for sep in [":", cue.split()[-1]]:
                        if sep.lower() in full_text.lower():
                            idx = full_text.lower().rfind(sep.lower()) + len(sep)
                            value = full_text[idx:].strip()
                            if value:
                                results[field_name] = value
                                break
                    if results[field_name]:
                        break
            if results[field_name]:
                break
    return results

extracted = extract_fields(ocr_data, SCHEMA, SIMULATION_MODE)

# ── Display Results ──
print("\n" + "=" * 64)
print("  EXTRACTED FIELDS (Schema-Driven)")
print("=" * 64)
for field_name, value in extracted.items():
    status = "✅" if value else "❌ NOT FOUND"
    print(f"  {field_name:<20} → {value or 'N/A':<25} {status}")
print("=" * 64)

# Provenance summary
log.info("Provenance: All extracted values derived from OCR tokens above "
         f"confidence threshold ({CONFIDENCE_THRESHOLD}). "
         "Low-confidence regions excluded from extraction.")
log.success("Document Intelligence pipeline complete — structured data ready for integration.")

---

## 4. Scientific Research Agent (§6.3)

Scientific Research agents integrate knowledge from multiple databases, reconcile
conflicting results, and produce synthesis reports that highlight **consensus,
divergence, and gaps** in knowledge (§6.3, p.20).

### The Three-Phase Research Workflow (Ref: §6.3, pp.21–25)

| Phase | Function | Output |
|-------|----------|--------|
| 1. **Broad Literature Scanning** | Query academic databases using semantic search | Raw corpus (DataFrame) |
| 2. **Thematic Clustering** | Generate embeddings, run KMeans, label clusters | Grouped papers with themes |
| 3. **Synthesis & Reporting** | Extractive summaries per cluster, evidence table | Actionable research map |

### The Cognitive Loop (Ref: §6.3, p.20)

- **Perception:** Receive research query as a multi-stage objective
- **Reasoning:** Translate into a broad semantic search strategy
- **Planning:** Multi-phase strategy across multiple knowledge sources
- **Action:** Query databases, cluster papers, traverse citation graphs
- **Learning:** Produce comparative tables and synthesis reports

### 4.1 Literature Scanning

Phase 1 queries the arXiv API for papers matching the research topic. In Simulation Mode,
`mock_search_arxiv()` returns 12 synthetic papers covering the topics discussed in §6.3:
RAG evaluation, domain-specific retrieval, citation graphs, provenance tracking, and
document intelligence.

In [ ]:
# ─────────────────────────────────────────────
# 4.1 Broad Literature Scanning
# Ref: §6.3 pp.22-23 — search_arxiv(), arxiv.Search()
# ─────────────────────────────────────────────

RESEARCH_QUERY = "large language models retrieval augmented generation evaluation"
MAX_RESULTS = 12
CLUSTERS = 4  # Ref: §6.3 p.22 — CLUSTERS = 4

@fail_gracefully(fallback_return=lambda: mock_search_arxiv(RESEARCH_QUERY), section_ref="6.3")
def search_literature(query, max_results, simulation_mode):
    if simulation_mode:
        log.info("[SIMULATION MODE] Using mock arXiv papers.")
        return mock_search_arxiv(query, max_results)
    else:
        import arxiv
        results = []
        for r in arxiv.Search(
            query=query,
            max_results=max_results,
            sort_by=arxiv.SortCriterion.Relevance,
        ).results():
            results.append({
                "title": r.title.strip(),
                "summary": r.summary.strip(),
                "authors": ", ".join(a.name for a in r.authors),
                "published": r.published.strftime("%Y-%m-%d"),
                "url": r.entry_id,
            })
        return pd.DataFrame(results)

df = search_literature(RESEARCH_QUERY, MAX_RESULTS, SIMULATION_MODE)

# Combine title + abstract as the unit of meaning (Ref: §6.3 p.23)
df["text"] = df["title"].astype(str) + ". " + df["summary"].astype(str)

print(f"\nPapers retrieved: {len(df)}")
print(f"Query: {RESEARCH_QUERY}")
print(f"\nFirst 3 titles:")
for i, row in df.head(3).iterrows():
    print(f"  {i+1}. {row['title']}")
    print(f"     {row['authors']} | {row['published']}")

### 4.2 Thematic Clustering

Phase 2 groups papers by shared themes (Ref: §6.3, pp.23–24). We generate embeddings
with `sentence-transformers` (`all-MiniLM-L6-v2`) and cluster using **KMeans** with `k=4`.

For each cluster, we:
1. Extract a label from the most frequent terms in nearby titles
2. Assemble an extractive summary from abstracts closest to the cluster centroid

In Simulation Mode with no model available, we fall back to simple hash-based
pseudo-embeddings.

In [ ]:
# ─────────────────────────────────────────────
# 4.2 Thematic Clustering
# Ref: §6.3 pp.23-24 — SentenceTransformer, KMeans, cluster labels
# ─────────────────────────────────────────────

import re
from collections import Counter
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

@fail_gracefully(fallback_return=None, section_ref="6.3")
def compute_embeddings(texts):
    """Try sentence-transformers; fall back to hash-based vectors."""
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer("all-MiniLM-L6-v2")
        emb = model.encode(texts, normalize_embeddings=True)
        log.success("Embeddings computed with all-MiniLM-L6-v2.")
        return emb
    except Exception as e:
        log.error(f"SentenceTransformer unavailable: {e}. Using hash-based fallback.")
        import hashlib
        vecs = []
        for t in texts:
            seed = int(hashlib.md5(t.encode()).hexdigest()[:8], 16) % (2**31)
            rng = np.random.RandomState(seed)
            v = rng.randn(128).astype(float)
            v = v / (np.linalg.norm(v) + 1e-10)
            vecs.append(v)
        return np.array(vecs)

emb = compute_embeddings(df["text"].tolist())

# Run KMeans clustering (Ref: §6.3 p.23 — CLUSTERS = 4)
kmeans = KMeans(n_clusters=CLUSTERS, n_init="auto", random_state=42)
labels = kmeans.fit_predict(emb)
df["cluster"] = labels

log.info(f"KMeans clustering complete: {CLUSTERS} clusters from {len(df)} papers.")
print(f"\nCluster distribution:")
for c in sorted(df["cluster"].unique()):
    count = (df["cluster"] == c).sum()
    print(f"  Cluster {c}: {count} papers")

In [ ]:
# ─────────────────────────────────────────────
# 4.3 Cluster Labeling & Extractive Summaries
# Ref: §6.3 pp.23-24 — label_cluster(), summarize_cluster()
# ─────────────────────────────────────────────

# Stopwords for label extraction (Ref: §6.3 p.23)
STOP_WORDS = set("""
the and for with from into using among toward towards on in of to via
model models data paper study approach results method methods novel new
large language based task tasks text query queries augment retrieval
""".split())

def clean_tokens(s):
    """Extract meaningful words from a title. Ref: §6.3 p.23."""
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s\-]", " ", s)
    toks = [t for t in s.split() if len(t) > 2]
    return [t for t in toks if t not in STOP_WORDS]

def label_cluster(c_idx, k=6, sample_n=6):
    """Generate a descriptive label from top terms in nearest titles."""
    centroid = kmeans.cluster_centers_[c_idx].reshape(1, -1)
    sims = cosine_similarity(emb, centroid).ravel()
    top_ids = sims.argsort()[-sample_n:]
    words = []
    for i in top_ids:
        words.extend(clean_tokens(df.iloc[i]["title"]))
    top = [w for w, _ in Counter(words).most_common(k)]
    return ", ".join(top) if top else "mixed theme"

def summarize_cluster(c_idx, n_sentences=3):
    """Extractive summary from abstracts closest to centroid. Ref: §6.3 p.24."""
    centroid = kmeans.cluster_centers_[c_idx].reshape(1, -1)
    sims = cosine_similarity(emb, centroid).ravel()
    top_ids = np.argsort(sims)[-6:]  # Top 6 most central docs
    candidates = []
    for i in top_ids:
        for s in re.split(r"(?<=[.!?])\s+", df.iloc[i]["summary"]):
            if 40 < len(s) < 300:
                candidates.append((s, i))
    if not candidates:
        return "Cluster summary not available."
    s_emb = compute_embeddings([s for s, _ in candidates])
    if s_emb is None:
        return candidates[0][0]
    s_sims = cosine_similarity(s_emb, centroid).ravel()
    top_s = np.argsort(s_sims)[-n_sentences:]
    return " ".join(candidates[i][0] for i in sorted(top_s))

# ── Generate Synthesis Report ──
print("\n" + "=" * 64)
print("  SCIENTIFIC RESEARCH AGENT — SYNTHESIS REPORT")
print(f"  Query: {RESEARCH_QUERY}")
print(f"  Papers: {len(df)} | Clusters: {CLUSTERS}")
print("=" * 64)

for c in sorted(df["cluster"].unique()):
    theme = label_cluster(c)
    summary = summarize_cluster(c)
    count = (df["cluster"] == c).sum()
    print(f"\n{'─' * 64}")
    print(f"  CLUSTER {c}: {theme}")
    print(f"  Papers: {count}")
    print(f"{'─' * 64}")
    print(f"  Synthesis: {summary}")
    print(f"\n  Representative papers:")
    reps = df[df["cluster"] == c].sort_values("published", ascending=False).head(3)
    for _, r in reps.iterrows():
        print(f"    • {r['title']}")
        print(f"      {r['authors']} | {r['published']}")
        print(f"      {r['url']}")

print("\n" + "=" * 64)
log.success("Synthesis report complete.")

### 4.4 Evidence Table

The final output is a structured evidence table — a concise reference that a researcher
can use to navigate the literature (Ref: §6.3, p.25). Each row links a paper to its
cluster, publication date, and URL.

In [ ]:
# ─────────────────────────────────────────────
# 4.4 Evidence Table
# Ref: §6.3 p.25 — "simple evidence table"
# ─────────────────────────────────────────────

evidence_table = (
    df.sort_values(["cluster", "published"], ascending=[True, False])
    [["cluster", "title", "authors", "published", "url"]]
    .reset_index(drop=True)
)

print("\n" + "=" * 64)
print("  EVIDENCE TABLE (all papers by cluster)")
print("=" * 64)
for c in sorted(df["cluster"].unique()):
    cluster_papers = evidence_table[evidence_table["cluster"] == c]
    theme = label_cluster(c)
    print(f"\n  Cluster {c}: {theme}")
    print("  " + "─" * 60)
    for _, row in cluster_papers.iterrows():
        print(f"    {row['published']} | {row['title'][:55]}...")
        print(f"    {row['url']}")
print("\n" + "=" * 64)

log.info(
    "Note (§6.3, p.27): In production, this evidence table would include "
    "quality signals, methodology tags, and citation counts per paper."
)

---

## 5. The Knowledge Agent Spectrum (§Summary)

The evolution from Knowledge Retrieval agents to Document Intelligence agents to
Scientific Research agents demonstrates **increasing levels of sophistication** in
agent engineering (§Summary, p.28). Each type represents different capability levels
of the cognitive loop.

### Comparison Table (Ref: Table 6.1, pp.29–30)

| Agent Type | Primary Role | Key Capabilities | Typical Use Cases | Capability Level |
|------------|-------------|-------------------|-------------------|-----------------|
| **Knowledge Retrieval** | Connect LLMs to live data sources | Mitigates cutoffs, implements RAG, structured + unstructured retrieval | Legal research, market intelligence, enterprise KB | Level 2–3: Tool-using to early planning |
| **Document Intelligence** | Convert unstructured docs to structured data | OCR + preprocessing, layout parsing, entity extraction, ERP/CRM integration | Healthcare claims, financial contracts, supply chain | Level 2–3: Tool-using to early planning |
| **Scientific Research** | Synthesize across multiple databases | Cluster + compare evidence, identify consensus/gaps, generate hypotheses | Drug discovery, climate science, policy analysis | Level 4: Learning agent |

### The Complete Knowledge Pipeline

Together, these agents form a pipeline from **discovery to decision-ready insights**:

1. **Knowledge Retrieval** agents *find* relevant information and ground it in sources
2. **Document Intelligence** agents *extract* structured data from within those documents
3. **Scientific Research** agents *synthesize* findings across corpora, revealing patterns and gaps

This progression from information access to knowledge creation represents a fundamental
shift in how we collaborate with AI — from static archives to dynamic, evidence-grounded
partners in discovery.

In [ ]:
# ─────────────────────────────────────────────
# 5.1 Agent Comparison — Programmatic Summary
# Ref: Table 6.1, pp.29-30
# ─────────────────────────────────────────────

comparison_df = pd.DataFrame({
    "Agent Type": [
        "Knowledge Retrieval",
        "Document Intelligence",
        "Scientific Research",
    ],
    "Primary Role": [
        "Connect LLMs to live, authoritative information",
        "Convert unstructured documents to structured data",
        "Synthesize information across multiple databases",
    ],
    "Capability Level": [
        "Level 2-3 (Tool-using → Planning)",
        "Level 2-3 (Tool-using → Planning)",
        "Level 4 (Learning agent)",
    ],
    "Key Section": ["§6.1", "§6.2", "§6.3"],
})

print("\n" + "=" * 64)
print("  KNOWLEDGE AGENT SPECTRUM — Chapter 6 Summary")
print("=" * 64)
print(comparison_df.to_string(index=False))
print("=" * 64)

log.success(
    "Chapter 6 complete. The three agent types form a pipeline from "
    "information retrieval → document understanding → research synthesis."
)
print("\n📘 Next: Chapter 7 explores deployment strategies for these agents at scale.")

---

**End of Chapter 6 Notebook**

**Author:** Imran Ahmad
**Book:** AI Agents (Packt, 2026)

For troubleshooting, see [troubleshooting.md](troubleshooting.md).
For AI assistant integration, see [AGENTS.md](AGENTS.md).